<a href="https://colab.research.google.com/github/henryzhou24/IDX-Exchange-Data-Analyst/blob/main/IDX_Weeks4_5_Data_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Weeks 4–5: Data Cleaning and Preparation

## Objective

This notebook prepares the IDX Exchange MLS datasets for reliable analysis by standardizing data types, reviewing missing values, identifying invalid numeric values, validating date consistency, and checking geographic coordinates.

## Input Data

- Residential Sold dataset with mortgage-rate enrichment
- Residential Listings dataset with mortgage-rate enrichment

## Data Integrity Principles

1. Preserve the original input files.
2. Document every transformation and the reason for performing it.
3. Record row counts before and after cleaning.
4. Flag questionable records before removing them.
5. Validate the final datasets and retain the `rate_30yr_fixed` field.

## Workflow

1. Load and inspect the datasets.
2. Audit columns, missing values, and data types.
3. Standardize date and numeric fields.
4. Review redundant columns and missing values.
5. Flag invalid numeric values.
6. Check date consistency.
7. Validate geographic coordinates.
8. Export cleaned datasets and quality reports.

In [1]:
from google.colab import drive
import os
import pandas as pd

# Mount Google Drive
drive.mount("/content/drive")

# Define project paths
BASE_DIR = "/content/drive/MyDrive/IDX-Exchange"
OUTPUT_DIR = os.path.join(BASE_DIR, "output")

SOLD_FILE = os.path.join(
    OUTPUT_DIR,
    "CRMLSSold_202401_202604_Residential_WithMortgageRate.csv"
)

LISTINGS_FILE = os.path.join(
    OUTPUT_DIR,
    "CRMLSListing_202401_202604_Residential_WithMortgageRate.csv"
)

# Confirm that both input files exist
for file_path in [SOLD_FILE, LISTINGS_FILE]:
    if os.path.exists(file_path):
        size_mb = os.path.getsize(file_path) / (1024 ** 2)
        print(f"FOUND: {os.path.basename(file_path)} ({size_mb:.1f} MB)")
    else:
        print(f"NOT FOUND: {file_path}")

Mounted at /content/drive
FOUND: CRMLSSold_202401_202604_Residential_WithMortgageRate.csv (231.4 MB)
FOUND: CRMLSListing_202401_202604_Residential_WithMortgageRate.csv (295.1 MB)


## Step 1: Initial Dataset Inspection

Before cleaning, we inspect the datasets to establish a baseline. This step records the original row counts, column names, data types, and missing-value counts.

We will use this baseline to determine which fields require conversion or further investigation and to verify the impact of subsequent cleaning operations.

No records or columns are removed in this step.

In [2]:
# Load the datasets without modifying the source files
sold = pd.read_csv(SOLD_FILE, low_memory=False)
listings = pd.read_csv(LISTINGS_FILE, low_memory=False)

datasets = {
    "Sold": sold,
    "Listings": listings
}

# Record baseline row and column counts
print("=== BASELINE DATASET SUMMARY ===")

for name, df in datasets.items():
    print(f"\n{name}")
    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns):,}")
    print(f"Duplicate column names: {df.columns.duplicated().sum():,}")
    print(f"Exact duplicate rows: {df.duplicated().sum():,}")

# Check whether the mortgage-rate column exists
print("\n=== MORTGAGE RATE CHECK ===")

for name, df in datasets.items():
    print(
        f"{name}: rate_30yr_fixed present = "
        f"{'rate_30yr_fixed' in df.columns}"
    )

=== BASELINE DATASET SUMMARY ===

Sold
Rows: 414,003
Columns: 86
Duplicate column names: 0
Exact duplicate rows: 0

Listings
Rows: 547,162
Columns: 86
Duplicate column names: 0
Exact duplicate rows: 0

=== MORTGAGE RATE CHECK ===
Sold: rate_30yr_fixed present = True
Listings: rate_30yr_fixed present = True


In [3]:
# Fields specifically required by the handbook
date_fields = [
    "CloseDate",
    "PurchaseContractDate",
    "ListingContractDate",
    "ContractStatusChangeDate"
]

numeric_fields = [
    "ClosePrice",
    "LivingArea",
    "DaysOnMarket",
    "Bedrooms",
    "Bathrooms",
    "Latitude",
    "Longitude",
    "latfilled",
    "lonfilled",
    "rate_30yr_fixed"
]

for name, df in datasets.items():
    print(f"\n{'=' * 60}")
    print(f"{name.upper()} — COLUMN AUDIT")
    print("=" * 60)

    print("\nDate fields:")
    for col in date_fields:
        if col in df.columns:
            print(f"  {col}: {df[col].dtype}")
        else:
            print(f"  {col}: NOT PRESENT")

    print("\nNumeric and geographic fields:")
    for col in numeric_fields:
        if col in df.columns:
            print(f"  {col}: {df[col].dtype}")
        else:
            print(f"  {col}: NOT PRESENT")

    print("\nFirst 30 column names:")
    print(df.columns.tolist()[:30])


SOLD — COLUMN AUDIT

Date fields:
  CloseDate: object
  PurchaseContractDate: object
  ListingContractDate: object
  ContractStatusChangeDate: object

Numeric and geographic fields:
  ClosePrice: float64
  LivingArea: float64
  DaysOnMarket: int64
  Bedrooms: NOT PRESENT
  Bathrooms: NOT PRESENT
  Latitude: float64
  Longitude: float64
  latfilled: object
  lonfilled: object
  rate_30yr_fixed: float64

First 30 column names:
['Flooring', 'ViewYN', 'WaterfrontYN', 'BasementYN', 'PoolPrivateYN', 'OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate', 'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude', 'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea', 'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName', 'CoListOfficeName', 'ListAgentFullName', 'CoListAgentFirstName', 'CoListAgentLastName', 'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName', 'FireplacesTotal', 'AssociationFeeFrequency']

LISTINGS — COLUMN AUDIT

Date fields:


## Step 2: Inspect Relevant MLS Fields

MLS datasets may use field names that differ from generic descriptions in the handbook. For example, bedroom and bathroom counts may be stored in fields such as `BedroomsTotal` or `BathroomsTotal`.

This section identifies the actual fields in each dataset and inspects representative values. We will use these findings to define the cleaning rules consistently across Sold and Listings records.

No data is modified in this step.

In [4]:
# Find columns related to dates, bedrooms, bathrooms, coordinates,
# and other fields relevant to cleaning.

search_terms = [
    "date",
    "bedroom",
    "bathroom",
    "latitude",
    "longitude",
    "latfilled",
    "lonfilled",
    "livingarea",
    "daysonmarket",
    "closeprice",
    "purchasecontract",
    "listingcontract",
    "contractstatuschange"
]

for name, df in datasets.items():
    print(f"\n{'=' * 65}")
    print(f"{name.upper()} — RELEVANT COLUMN NAMES")
    print("=" * 65)

    for term in search_terms:
        matches = [
            col for col in df.columns
            if term in col.lower()
        ]
        if matches:
            print(f"{term}: {matches}")

    # Inspect selected geographic columns
    geo_cols = [
        col for col in [
            "Latitude", "Longitude", "latfilled", "lonfilled"
        ]
        if col in df.columns
    ]

    print("\nGeographic field examples:")
    if geo_cols:
        print(df[geo_cols].head(10).to_string(index=False))

    # Inspect actual date values before conversion
    actual_date_cols = [
        col for col in [
            "CloseDate",
            "PurchaseContractDate",
            "ListingContractDate",
            "ContractStatusChangeDate"
        ]
        if col in df.columns
    ]

    print("\nDate field examples:")
    print(df[actual_date_cols].head(5).to_string(index=False))


SOLD — RELEVANT COLUMN NAMES
date: ['CloseDate', 'ContractStatusChangeDate', 'PurchaseContractDate', 'ListingContractDate']
bedroom: ['BedroomsTotal', 'MainLevelBedrooms']
bathroom: ['BathroomsTotalInteger']
latitude: ['Latitude']
longitude: ['Longitude']
latfilled: ['latfilled']
lonfilled: ['lonfilled']
livingarea: ['LivingArea']
daysonmarket: ['DaysOnMarket']
closeprice: ['ClosePrice']
purchasecontract: ['PurchaseContractDate']
listingcontract: ['ListingContractDate']
contractstatuschange: ['ContractStatusChangeDate']

Geographic field examples:
 Latitude   Longitude latfilled lonfilled
37.566106 -122.327954      True      True
32.659315 -117.096922      True      True
32.659284 -117.097174      True      True
35.277199 -120.646786     False     False
39.419080 -123.814842     False     False
37.325454 -121.780303      True      True
32.715089 -117.170568     False     False
34.070215 -118.258618     False     False
34.107850 -118.219273     False     False
37.589762 -122.399887    

## Step 3: Audit Potentially Redundant Columns

During the initial inspection, several Listings columns were found with `.1` suffixes. These columns may be duplicated fields created during an earlier data-processing step, but we must verify that before removing them.

For each matching pair, we compare missing-value patterns and values row by row. We also inspect the percentage of missing values in key cleaning fields.

This is an audit only. No columns or records are removed.

In [5]:
# Compare columns such as Latitude and Latitude.1.
# A suffix alone is not enough to justify removing a column.

for name, df in datasets.items():
    print(f"\n{'=' * 70}")
    print(f"{name.upper()} — POTENTIAL DUPLICATE COLUMN AUDIT")
    print("=" * 70)

    suffix_cols = [col for col in df.columns if col.endswith(".1")]

    if not suffix_cols:
        print("No columns ending in '.1' found.")
        continue

    for duplicate_col in suffix_cols:
        base_col = duplicate_col[:-2]

        if base_col not in df.columns:
            print(f"\n{duplicate_col}: no matching base column")
            continue

        left = df[base_col]
        right = df[duplicate_col]

        both_present = left.notna() & right.notna()
        either_present = left.notna() | right.notna()

        matching = (
            left.eq(right) | (left.isna() & right.isna())
        )

        comparable_count = int(both_present.sum())
        mismatch_count = int(
            (both_present & ~left.eq(right)).sum()
        )

        print(f"\nPair: {base_col} vs {duplicate_col}")
        print(f"  Base column missing: {left.isna().sum():,}")
        print(f"  Suffix column missing: {right.isna().sum():,}")
        print(f"  Both values present: {comparable_count:,}")
        print(f"  Both present but different: {mismatch_count:,}")
        print(f"  Equal including matching missing values: "
              f"{int(matching.sum()):,} / {len(df):,}")

        if mismatch_count > 0:
            print("  Examples of differences:")
            examples = df.loc[
                both_present & ~left.eq(right),
                [base_col, duplicate_col]
            ].head(5)
            print(examples.to_string(index=False))

# Inspect missingness in fields relevant to the cleaning task.
key_fields = [
    "CloseDate",
    "PurchaseContractDate",
    "ListingContractDate",
    "ContractStatusChangeDate",
    "ClosePrice",
    "ListPrice",
    "LivingArea",
    "DaysOnMarket",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "Latitude",
    "Longitude",
    "rate_30yr_fixed"
]

for name, df in datasets.items():
    print(f"\n{'=' * 70}")
    print(f"{name.upper()} — KEY FIELD COMPLETENESS")
    print("=" * 70)

    rows = []
    for col in key_fields:
        if col in df.columns:
            missing_count = int(df[col].isna().sum())
            rows.append({
                "column": col,
                "dtype": str(df[col].dtype),
                "missing_count": missing_count,
                "missing_percent": round(
                    missing_count / len(df) * 100, 2
                )
            })

    completeness = pd.DataFrame(rows)
    print(completeness.to_string(index=False))


SOLD — POTENTIAL DUPLICATE COLUMN AUDIT
No columns ending in '.1' found.

LISTINGS — POTENTIAL DUPLICATE COLUMN AUDIT

Pair: PropertyType vs PropertyType.1
  Base column missing: 0
  Suffix column missing: 0
  Both values present: 547,162
  Both present but different: 0
  Equal including matching missing values: 547,162 / 547,162

Pair: ListAgentFirstName vs ListAgentFirstName.1
  Base column missing: 4,354
  Suffix column missing: 4,354
  Both values present: 542,808
  Both present but different: 0
  Equal including matching missing values: 547,162 / 547,162

Pair: DaysOnMarket vs DaysOnMarket.1
  Base column missing: 0
  Suffix column missing: 0
  Both values present: 547,162
  Both present but different: 0
  Equal including matching missing values: 547,162 / 547,162

Pair: LivingArea vs LivingArea.1
  Base column missing: 552
  Suffix column missing: 552
  Both values present: 546,610
  Both present but different: 0
  Equal including matching missing values: 547,162 / 547,162

Pair

## Step 4: Investigate Column Differences

Most `.1` columns in the Listings dataset are exact duplicates. Two exceptions need special handling:

- `BuyerOfficeName` has one conflicting non-missing value.
- `UnparsedAddress` has one row where the original and suffixed columns differ in missingness.

This step identifies those records using `ListingKey` where available. We will use the results to decide how to preserve information before dropping redundant columns.

In [6]:
# Inspect the one BuyerOfficeName conflict.
print("=== BUYER OFFICE NAME CONFLICT ===")

buyer_conflict = (
    listings["BuyerOfficeName"].notna()
    & listings["BuyerOfficeName.1"].notna()
    & listings["BuyerOfficeName"].ne(listings["BuyerOfficeName.1"])
)

buyer_cols = [
    col for col in [
        "ListingKey",
        "PropertyType",
        "CloseDate",
        "BuyerOfficeName",
        "BuyerOfficeName.1",
    ]
    if col in listings.columns
]

print(f"Conflicting rows: {int(buyer_conflict.sum()):,}")
print(listings.loc[buyer_conflict, buyer_cols].to_string(index=False))


# Inspect rows where UnparsedAddress and UnparsedAddress.1
# have different missing-value patterns.
print("\n=== UNPARSED ADDRESS MISSINGNESS DIFFERENCE ===")

address_difference = (
    listings["UnparsedAddress"].isna()
    != listings["UnparsedAddress.1"].isna()
)

address_cols = [
    col for col in [
        "ListingKey",
        "PropertyType",
        "UnparsedAddress",
        "UnparsedAddress.1",
        "Latitude",
        "Longitude",
    ]
    if col in listings.columns
]

print(f"Rows with different missingness: {int(address_difference.sum()):,}")
print(
    listings.loc[address_difference, address_cols]
    .to_string(index=False)
)

=== BUYER OFFICE NAME CONFLICT ===
Conflicting rows: 1
 ListingKey PropertyType CloseDate BuyerOfficeName BuyerOfficeName.1
 1151935665  Residential       NaN         Compass                Co

=== UNPARSED ADDRESS MISSINGNESS DIFFERENCE ===
Rows with different missingness: 1
 ListingKey PropertyType UnparsedAddress UnparsedAddress.1  Latitude   Longitude
 1151935665  Residential 1369 Milton Ave               NaN  37.88143 -122.044742


## Step 5: Remove Redundant Columns Without Losing Information

The audit confirmed that most suffixed columns duplicate their original columns exactly. We will:

1. Create independent working copies of Sold and Listings.
2. Remove the nine verified duplicate columns.
3. Keep `UnparsedAddress` as the canonical address column, checking the suffix for any values that might need to be recovered.
4. Rename `BuyerOfficeName.1` to `BuyerOfficeName_suffix_review` so the conflicting value remains available for investigation.
5. Confirm that row counts and the mortgage-rate field are unchanged.

No records will be deleted in this step.

In [7]:
# Create working copies. The original DataFrames remain unchanged.
sold_clean = sold.copy()
listings_clean = listings.copy()

# These nine columns were confirmed to match their base columns
# across all rows, including missing values.
verified_duplicate_columns = [
    "PropertyType.1",
    "ListAgentFirstName.1",
    "DaysOnMarket.1",
    "LivingArea.1",
    "Longitude.1",
    "Latitude.1",
    "ListPrice.1",
    "ListAgentLastName.1",
    "CloseDate.1",
]

# Remove only the verified duplicates that exist.
columns_to_drop = [
    col for col in verified_duplicate_columns
    if col in listings_clean.columns
]
listings_clean = listings_clean.drop(columns=columns_to_drop)

# Keep the canonical address. Recover a value from the suffix only
# if the canonical address is missing and the suffix has a value.
if "UnparsedAddress.1" in listings_clean.columns:
    address_suffix = listings_clean["UnparsedAddress.1"]

    recover_address = (
        listings_clean["UnparsedAddress"].isna()
        & address_suffix.notna()
    )

    recovered_count = int(recover_address.sum())

    listings_clean.loc[
        recover_address, "UnparsedAddress"
    ] = address_suffix.loc[recover_address]

    listings_clean = listings_clean.drop(
        columns=["UnparsedAddress.1"]
    )
else:
    recovered_count = 0

# Preserve the conflicting BuyerOfficeName value for review.
if "BuyerOfficeName.1" in listings_clean.columns:
    listings_clean = listings_clean.rename(
        columns={
            "BuyerOfficeName.1": "BuyerOfficeName_suffix_review"
        }
    )

# Verify row counts, columns, and mortgage-rate retention.
print("=== COLUMN CLEANUP VALIDATION ===")

for name, df in [
    ("Sold", sold_clean),
    ("Listings", listings_clean),
]:
    original = datasets[name]

    print(f"\n{name}")
    print(f"Original rows: {len(original):,}")
    print(f"Working-copy rows: {len(df):,}")
    print(f"Original columns: {len(original.columns):,}")
    print(f"Working-copy columns: {len(df.columns):,}")
    print(f"Row count unchanged: {len(original) == len(df)}")
    print(f"Mortgage-rate field retained: {'rate_30yr_fixed' in df.columns}")

print(f"\nRedundant columns removed: {len(columns_to_drop) + 1}")
print(f"Address values recovered from suffix: {recovered_count}")
print(
    "Buyer office review column retained:",
    "BuyerOfficeName_suffix_review" in listings_clean.columns
)
print(
    "Buyer office conflict record:",
    listings_clean.loc[
        listings_clean["ListingKey"].astype(str) == "1151935665",
        [
            "ListingKey",
            "BuyerOfficeName",
            "BuyerOfficeName_suffix_review",
            "UnparsedAddress",
        ],
    ].to_string(index=False)
)

=== COLUMN CLEANUP VALIDATION ===

Sold
Original rows: 414,003
Working-copy rows: 414,003
Original columns: 86
Working-copy columns: 86
Row count unchanged: True
Mortgage-rate field retained: True

Listings
Original rows: 547,162
Working-copy rows: 547,162
Original columns: 86
Working-copy columns: 76
Row count unchanged: True
Mortgage-rate field retained: True

Redundant columns removed: 10
Address values recovered from suffix: 0
Buyer office review column retained: True
Buyer office conflict record:  ListingKey BuyerOfficeName BuyerOfficeName_suffix_review UnparsedAddress
 1151935665         Compass                            Co 1369 Milton Ave


## Step 6: Standardize Data Types

Consistent data types are essential for accurate calculations, date comparisons, and visualizations.

In this step, I will:

- Convert `CloseDate`, `PurchaseContractDate`, `ListingContractDate`, and `ContractStatusChangeDate` to pandas datetime types.
- Convert key price, property-size, bedroom, bathroom, days-on-market, coordinate, and mortgage-rate fields to numeric types.
- Record original missing values separately from non-missing values that fail conversion.
- Preserve all rows and retain the original input DataFrames.

I will not fill missing values or remove records in this step. Any conversion failures will be documented for review.

In [8]:
# Fields that should represent dates.
date_fields = [
    "CloseDate",
    "PurchaseContractDate",
    "ListingContractDate",
    "ContractStatusChangeDate",
]

# Fields used in analysis and quality checks.
numeric_fields = [
    "ClosePrice",
    "ListPrice",
    "LivingArea",
    "DaysOnMarket",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "Latitude",
    "Longitude",
    "rate_30yr_fixed",
]

conversion_log = []

for dataset_name, df in [
    ("Sold", sold_clean),
    ("Listings", listings_clean),
]:
    print(f"\n{'=' * 70}")
    print(f"{dataset_name.upper()} — TYPE STANDARDIZATION")
    print("=" * 70)

    # Convert date fields.
    for col in date_fields:
        if col not in df.columns:
            continue

        original = df[col]
        original_nonmissing = original.notna()

        converted = pd.to_datetime(original, errors="coerce")

        missing_before = int(original.isna().sum())
        conversion_failures = int(
            (original_nonmissing & converted.isna()).sum()
        )

        df[col] = converted

        conversion_log.append({
            "dataset": dataset_name,
            "column": col,
            "target_type": "datetime",
            "missing_before": missing_before,
            "conversion_failures": conversion_failures,
            "missing_after": int(df[col].isna().sum()),
            "final_dtype": str(df[col].dtype),
        })

    # Convert numeric fields.
    for col in numeric_fields:
        if col not in df.columns:
            continue

        original = df[col]
        original_nonmissing = original.notna()

        converted = pd.to_numeric(original, errors="coerce")

        missing_before = int(original.isna().sum())
        conversion_failures = int(
            (original_nonmissing & converted.isna()).sum()
        )

        df[col] = converted

        conversion_log.append({
            "dataset": dataset_name,
            "column": col,
            "target_type": "numeric",
            "missing_before": missing_before,
            "conversion_failures": conversion_failures,
            "missing_after": int(df[col].isna().sum()),
            "final_dtype": str(df[col].dtype),
        })

# Display a detailed conversion summary.
conversion_summary = pd.DataFrame(conversion_log)

print("\n=== CONVERSION SUMMARY ===")
print(conversion_summary.to_string(index=False))

print("\n=== FINAL DATE AND NUMERIC DTYPES ===")
for dataset_name, df in [
    ("Sold", sold_clean),
    ("Listings", listings_clean),
]:
    print(f"\n{dataset_name}")
    for col in date_fields + numeric_fields:
        if col in df.columns:
            print(f"{col}: {df[col].dtype}")

print("\n=== VALIDATION ===")
for dataset_name, df in [
    ("Sold", sold_clean),
    ("Listings", listings_clean),
]:
    print(
        f"{dataset_name}: {len(df):,} rows; "
        f"rate_30yr_fixed present = {'rate_30yr_fixed' in df.columns}"
    )

print(
    "\nTotal conversion failures:",
    f"{conversion_summary['conversion_failures'].sum():,}"
)


SOLD — TYPE STANDARDIZATION

LISTINGS — TYPE STANDARDIZATION

=== CONVERSION SUMMARY ===
 dataset                   column target_type  missing_before  conversion_failures  missing_after    final_dtype
    Sold                CloseDate    datetime               0                    0              0 datetime64[ns]
    Sold     PurchaseContractDate    datetime             194                    0            194 datetime64[ns]
    Sold      ListingContractDate    datetime               1                    0              1 datetime64[ns]
    Sold ContractStatusChangeDate    datetime             594                    0            594 datetime64[ns]
    Sold               ClosePrice     numeric               0                    0              0        float64
    Sold                ListPrice     numeric               0                    0              0        float64
    Sold               LivingArea     numeric             234                    0            234        float64
    So

## Step 7: Identify Invalid Numeric Values

This step checks the key numeric fields against basic validity rules:

- `ClosePrice` must be greater than zero when present.
- `LivingArea` must be greater than zero when present.
- `DaysOnMarket` must not be negative.
- `BedroomsTotal` and `BathroomsTotalInteger` must not be negative.

For each rule, we will add a Boolean flag column. Missing values will not be classified as invalid by these flags; missingness will be summarized separately.

We will retain all records, including those with flagged values, to preserve data for review and document the quality issues transparently.

In [9]:
numeric_flag_summary = []

def add_numeric_quality_flags(df, dataset_name):
    """Add Boolean flags for invalid values and summarize each rule."""
    rules = {
        "ClosePrice": ("closeprice_nonpositive_flag", lambda s: s.notna() & (s <= 0)),
        "LivingArea": ("livingarea_nonpositive_flag", lambda s: s.notna() & (s <= 0)),
        "DaysOnMarket": ("daysonmarket_negative_flag", lambda s: s.notna() & (s < 0)),
        "BedroomsTotal": ("bedrooms_negative_flag", lambda s: s.notna() & (s < 0)),
        "BathroomsTotalInteger": ("bathrooms_negative_flag", lambda s: s.notna() & (s < 0)),
    }

    for col, (flag_col, condition) in rules.items():
        if col not in df.columns:
            print(f"{dataset_name}: {col} not found; skipped.")
            continue

        df[flag_col] = condition(df[col]).astype(bool)

        numeric_flag_summary.append({
            "dataset": dataset_name,
            "field": col,
            "flag_column": flag_col,
            "flagged_rows": int(df[flag_col].sum()),
            "missing_values": int(df[col].isna().sum()),
        })


for dataset_name, df in [
    ("Sold", sold_clean),
    ("Listings", listings_clean),
]:
    add_numeric_quality_flags(df, dataset_name)

numeric_quality_summary = pd.DataFrame(numeric_flag_summary)

print("=== NUMERIC QUALITY FLAG SUMMARY ===")
print(numeric_quality_summary.to_string(index=False))

# Summarize records with one or more numeric flags.
numeric_flag_columns = numeric_quality_summary["flag_column"].tolist()

print("\n=== RECORD-LEVEL SUMMARY ===")
for dataset_name, df in [
    ("Sold", sold_clean),
    ("Listings", listings_clean),
]:
    available_flags = [
        col for col in numeric_flag_columns if col in df.columns
    ]

    any_flagged = df[available_flags].any(axis=1)

    print(f"\n{dataset_name}")
    print(f"Rows: {len(df):,}")
    print(f"Rows with one or more numeric flags: {int(any_flagged.sum()):,}")
    print(f"Rows retained: {len(df):,}")

=== NUMERIC QUALITY FLAG SUMMARY ===
 dataset                 field                 flag_column  flagged_rows  missing_values
    Sold            ClosePrice closeprice_nonpositive_flag             0               0
    Sold            LivingArea livingarea_nonpositive_flag           150             234
    Sold          DaysOnMarket  daysonmarket_negative_flag            48               0
    Sold         BedroomsTotal      bedrooms_negative_flag             0              11
    Sold BathroomsTotalInteger     bathrooms_negative_flag             0              69
Listings            ClosePrice closeprice_nonpositive_flag             0          409481
Listings            LivingArea livingarea_nonpositive_flag           360             552
Listings          DaysOnMarket  daysonmarket_negative_flag            27               0
Listings         BedroomsTotal      bedrooms_negative_flag             0             133
Listings BathroomsTotalInteger     bathrooms_negative_flag             0 

**Sold**: 198 rows have at least one numeric quality flag.



**Listings**: 387 rows have at least one numeric quality flag.



**Negative DaysOnMarket**: 48 Sold rows and 27 Listings rows.



**Nonpositive LivingArea**: 150 Sold rows and 360 Listings rows.



No nonpositive ClosePrice or negative bedroom/bathroom values were found.



All rows were retained.

## Step 8: Date Consistency Checks

Create three Boolean flags:
- `listing_after_close_flag`: listing contract date occurs after closing.
- `purchase_after_close_flag`: purchase contract date occurs after closing.
- `negative_timeline_flag`: listing contract date occurs after purchase contract date, or purchase contract date occurs after closing.

Rows with missing dates are not flagged by comparisons that require those dates. Missing dates will be reported separately. No rows will be removed.

In [10]:
date_flag_summary = []

def add_date_flags(df, dataset_name):
    listing = df["ListingContractDate"]
    purchase = df["PurchaseContractDate"]
    close = df["CloseDate"]

    df["listing_after_close_flag"] = (
        listing.notna() & close.notna() & (listing > close)
    )

    df["purchase_after_close_flag"] = (
        purchase.notna() & close.notna() & (purchase > close)
    )

    listing_after_purchase = (
        listing.notna() & purchase.notna() & (listing > purchase)
    )
    purchase_after_close = (
        purchase.notna() & close.notna() & (purchase > close)
    )

    df["negative_timeline_flag"] = (
        listing_after_purchase | purchase_after_close
    )

    flags = [
        "listing_after_close_flag",
        "purchase_after_close_flag",
        "negative_timeline_flag",
    ]

    for flag in flags:
        date_flag_summary.append({
            "dataset": dataset_name,
            "flag": flag,
            "flagged_rows": int(df[flag].sum()),
        })

    print(f"\n{dataset_name}")
    print(f"Rows: {len(df):,}")
    print(f"Missing ListingContractDate: {listing.isna().sum():,}")
    print(f"Missing PurchaseContractDate: {purchase.isna().sum():,}")
    print(f"Missing CloseDate: {close.isna().sum():,}")
    print(
        "Rows with any date flag:",
        f"{int(df[flags].any(axis=1).sum()):,}"
    )


for dataset_name, df in [
    ("Sold", sold_clean),
    ("Listings", listings_clean),
]:
    add_date_flags(df, dataset_name)

date_consistency_summary = pd.DataFrame(date_flag_summary)

print("\n=== DATE CONSISTENCY SUMMARY ===")
print(date_consistency_summary.to_string(index=False))


Sold
Rows: 414,003
Missing ListingContractDate: 1
Missing PurchaseContractDate: 194
Missing CloseDate: 0
Rows with any date flag: 507

Listings
Rows: 547,162
Missing ListingContractDate: 0
Missing PurchaseContractDate: 285,676
Missing CloseDate: 388,096
Rows with any date flag: 530

=== DATE CONSISTENCY SUMMARY ===
 dataset                      flag  flagged_rows
    Sold  listing_after_close_flag            61
    Sold purchase_after_close_flag           239
    Sold    negative_timeline_flag           507
Listings  listing_after_close_flag            73
Listings purchase_after_close_flag           262
Listings    negative_timeline_flag           530


- Sold: 507 rows have at least one date-consistency flag.
- Listings: 530 rows have at least one date-consistency flag.
- listing_after_close_flag: 61 Sold and 73 Listings rows.
- purchase_after_close_flag: 239 Sold and 262 Listings rows.
- Missing dates are common in Listings: 285,676 lack PurchaseContractDate, and 388,096 lack CloseDate. These are not automatically errors.

The flags can overlap, so counts should not be added together. These records remain in the datasets for review.

## Step 9: Geographic Quality Checks

Validate `Latitude` and `Longitude` for missing values, zero sentinels, positive longitude, globally impossible coordinate ranges, and locations outside a broad California bounding box (latitude 32.5–42.1, longitude −124.5 to −114.0).

The checks add Boolean flags and summarize their counts. No rows are removed.

In [11]:
geographic_summary = []

def add_geographic_flags(df, dataset_name):
    lat = df["Latitude"]
    lon = df["Longitude"]

    df["coordinates_missing_flag"] = lat.isna() | lon.isna()
    df["latitude_zero_flag"] = lat.notna() & lat.eq(0)
    df["longitude_zero_flag"] = lon.notna() & lon.eq(0)
    df["longitude_positive_flag"] = lon.notna() & lon.gt(0)

    df["coordinates_out_of_range_flag"] = (
        (lat.notna() & ~lat.between(-90, 90))
        | (lon.notna() & ~lon.between(-180, 180))
    )

    # Broad California bounding box; outside does not always mean invalid.
    df["coordinates_outside_ca_flag"] = (
        lat.notna() & lon.notna()
        & ~lat.between(32.5, 42.1)
        | lat.notna() & lon.notna()
        & ~lon.between(-124.5, -114.0)
    )

    flag_columns = [
        "coordinates_missing_flag",
        "latitude_zero_flag",
        "longitude_zero_flag",
        "longitude_positive_flag",
        "coordinates_out_of_range_flag",
        "coordinates_outside_ca_flag",
    ]

    for flag in flag_columns:
        geographic_summary.append({
            "dataset": dataset_name,
            "flag": flag,
            "flagged_rows": int(df[flag].sum()),
        })

    print(f"\n{dataset_name}")
    print(f"Rows: {len(df):,}")
    print(f"Rows with any geographic flag: {int(df[flag_columns].any(axis=1).sum()):,}")


for dataset_name, df in [
    ("Sold", sold_clean),
    ("Listings", listings_clean),
]:
    add_geographic_flags(df, dataset_name)

geographic_quality_summary = pd.DataFrame(geographic_summary)

print("\n=== GEOGRAPHIC QUALITY SUMMARY ===")
print(geographic_quality_summary.to_string(index=False))


Sold
Rows: 414,003
Rows with any geographic flag: 4,195

Listings
Rows: 547,162
Rows with any geographic flag: 80,558

=== GEOGRAPHIC QUALITY SUMMARY ===
 dataset                          flag  flagged_rows
    Sold      coordinates_missing_flag          4104
    Sold            latitude_zero_flag            27
    Sold           longitude_zero_flag            27
    Sold       longitude_positive_flag            31
    Sold coordinates_out_of_range_flag             2
    Sold   coordinates_outside_ca_flag            91
Listings      coordinates_missing_flag         80247
Listings            latitude_zero_flag            64
Listings           longitude_zero_flag            64
Listings       longitude_positive_flag            73
Listings coordinates_out_of_range_flag             4
Listings   coordinates_outside_ca_flag           311


IDX Exchange — Weeks 4–5 Data Cleaning

1. Baseline and data integrity
- Sold: 414,003 rows, initially 86 columns.
- Listings: 547,162 rows, initially 86 columns.
- No exact duplicate rows or duplicate column names in either dataset.
- rate_30yr_fixed retained in both datasets.

2. Redundant columns
- Listings reduced from 86 to 76 columns by removing verified duplicate columns and handling the address suffix.
- One BuyerOfficeName conflict was retained for review: Compass vs. Co.
- No rows removed.

3. Data types and missing values
- Four date fields converted to datetime; zero conversion failures.
- Key numeric fields confirmed numeric; zero conversion failures.
- Existing missing values preserved. Missing CloseDate and PurchaseContractDate in Listings were not automatically treated as errors.

4. Numeric quality flags
- Sold: 198 rows with at least one numeric flag.
- Listings: 387 rows with at least one numeric flag.
- Negative DaysOnMarket: Sold 48; Listings 27.
- Nonpositive LivingArea: Sold 150; Listings 360.
- No nonpositive ClosePrice or negative bedroom/bathroom values found.

5. Date consistency flags
- Sold: 507 rows with at least one date flag.
- Listings: 530 rows with at least one date flag.
- ListingContractDate after CloseDate: Sold 61; Listings 73.
- PurchaseContractDate after CloseDate: Sold 239; Listings 262.
- Flag counts overlap and should not be added together.

6. Geographic quality flags
- Sold: 4,195 rows with at least one geographic flag.
- Listings: 80,558 rows with at least one geographic flag.
- Missing coordinates: Sold 4,104 (0.99%); Listings 80,247 (14.67%).
- Positive longitude: Sold 31; Listings 73.
- Coordinates outside broad California bounding box: Sold 91; Listings 311.
- Globally out-of-range coordinates: Sold 2; Listings 4.
- Geographic flags can overlap; outside the bounding box does not necessarily prove a record is invalid.

7. Data preservation
- No rows removed in these steps.
- Original input DataFrames and source files remain unchanged.
- Cleaned working copies retain rate_30yr_fixed.

Next: perform final validation, create a consolidated quality report, and export cleaned datasets and the documented Python script.

In [12]:
# Step 10: Final validation and consolidated report preparation

import pandas as pd
import os

quality_rows = []

original_datasets = {
    "Sold": sold,
    "Listings": listings,
}

cleaned_datasets = {
    "Sold": sold_clean,
    "Listings": listings_clean,
}

# Recalculate geographic flags with explicit logical grouping.
for name, df in cleaned_datasets.items():
    lat = df["Latitude"]
    lon = df["Longitude"]

    df["coordinates_outside_ca_flag"] = (
        lat.notna()
        & lon.notna()
        & (
            ~lat.between(32.5, 42.1)
            | ~lon.between(-124.5, -114.0)
        )
    )

# Add a consistent row format to the consolidated report.
def add_report_row(dataset, category, check, value, notes=""):
    quality_rows.append({
        "dataset": dataset,
        "category": category,
        "check": check,
        "value": value,
        "notes": notes,
    })

numeric_flags = [
    "closeprice_nonpositive_flag",
    "livingarea_nonpositive_flag",
    "daysonmarket_negative_flag",
    "bedrooms_negative_flag",
    "bathrooms_negative_flag",
]

date_flags = [
    "listing_after_close_flag",
    "purchase_after_close_flag",
    "negative_timeline_flag",
]

geo_flags = [
    "coordinates_missing_flag",
    "latitude_zero_flag",
    "longitude_zero_flag",
    "longitude_positive_flag",
    "coordinates_out_of_range_flag",
    "coordinates_outside_ca_flag",
]

for name, df in cleaned_datasets.items():
    original = original_datasets[name]

    # Core integrity checks.
    add_report_row(
        name, "Validation", "Original row count",
        len(original)
    )
    add_report_row(
        name, "Validation", "Cleaned row count",
        len(df), "Rows retained; no row-level filtering performed."
    )
    add_report_row(
        name, "Validation", "Row count unchanged",
        len(original) == len(df)
    )
    add_report_row(
        name, "Validation", "Cleaned column count",
        len(df.columns)
    )
    add_report_row(
        name, "Validation", "Duplicate column names",
        int(df.columns.duplicated().sum())
    )
    add_report_row(
        name, "Validation", "Exact duplicate rows",
        int(df.duplicated().sum())
    )
    add_report_row(
        name, "Validation", "Mortgage rate field present",
        "rate_30yr_fixed" in df.columns
    )
    add_report_row(
        name, "Validation", "Missing mortgage rate values",
        int(df["rate_30yr_fixed"].isna().sum())
    )

    # Missingness in important source fields.
    for col in [
        "CloseDate", "PurchaseContractDate", "ListingContractDate",
        "ContractStatusChangeDate", "ClosePrice", "ListPrice",
        "LivingArea", "DaysOnMarket", "BedroomsTotal",
        "BathroomsTotalInteger", "Latitude", "Longitude",
        "rate_30yr_fixed",
    ]:
        if col in df.columns:
            missing = int(df[col].isna().sum())
            add_report_row(
                name, "Missingness", col, missing,
                f"{missing / len(df) * 100:.2f}% of rows missing."
            )

    # Numeric quality flags.
    for flag in numeric_flags:
        if flag in df.columns:
            add_report_row(
                name, "Numeric quality", flag,
                int(df[flag].sum()),
                "Flagged rows; records retained."
            )

    # Date consistency flags.
    for flag in date_flags:
        if flag in df.columns:
            add_report_row(
                name, "Date consistency", flag,
                int(df[flag].sum()),
                "Flagged rows; counts may overlap."
            )

    # Geographic quality flags.
    for flag in geo_flags:
        if flag in df.columns:
            add_report_row(
                name, "Geographic quality", flag,
                int(df[flag].sum()),
                "Flagged rows; counts may overlap."
            )

    # Required dtype validation.
    for col in [
        "CloseDate", "PurchaseContractDate",
        "ListingContractDate", "ContractStatusChangeDate",
    ]:
        if col in df.columns:
            add_report_row(
                name, "Data types", f"{col} dtype",
                str(df[col].dtype),
                "Expected datetime dtype."
            )

    # Record numeric flag counts at row level.
    available_numeric_flags = [
        col for col in numeric_flags if col in df.columns
    ]
    available_date_flags = [
        col for col in date_flags if col in df.columns
    ]
    available_geo_flags = [
        col for col in geo_flags if col in df.columns
    ]

    add_report_row(
        name, "Record-level summary",
        "Rows with any numeric flag",
        int(df[available_numeric_flags].any(axis=1).sum())
    )
    add_report_row(
        name, "Record-level summary",
        "Rows with any date flag",
        int(df[available_date_flags].any(axis=1).sum())
    )
    add_report_row(
        name, "Record-level summary",
        "Rows with any geographic flag",
        int(df[available_geo_flags].any(axis=1).sum())
    )

quality_report = pd.DataFrame(quality_rows)

# Show concise validation results.
print("=== FINAL VALIDATION ===")
for name, df in cleaned_datasets.items():
    print(f"\n{name}")
    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns):,}")
    print(f"Duplicate column names: {df.columns.duplicated().sum():,}")
    print(f"Exact duplicate rows: {df.duplicated().sum():,}")
    print(f"Mortgage rate present: {'rate_30yr_fixed' in df.columns}")
    print(f"Missing mortgage rates: {df['rate_30yr_fixed'].isna().sum():,}")
    print(
        "Rows with numeric flags:",
        f"{int(df[available_numeric_flags].any(axis=1).sum()):,}"
    )

print("\n=== CONSOLIDATED REPORT PREVIEW ===")
print(f"Report rows: {len(quality_report):,}")
print(quality_report.head(15).to_string(index=False))

=== FINAL VALIDATION ===

Sold
Rows: 414,003
Columns: 100
Duplicate column names: 0
Exact duplicate rows: 0
Mortgage rate present: True
Missing mortgage rates: 0
Rows with numeric flags: 198

Listings
Rows: 547,162
Columns: 90
Duplicate column names: 0
Exact duplicate rows: 0
Mortgage rate present: True
Missing mortgage rates: 0
Rows with numeric flags: 387

=== CONSOLIDATED REPORT PREVIEW ===
Report rows: 84
dataset    category                        check  value                                            notes
   Sold  Validation           Original row count 414003                                                 
   Sold  Validation            Cleaned row count 414003 Rows retained; no row-level filtering performed.
   Sold  Validation          Row count unchanged   True                                                 
   Sold  Validation         Cleaned column count    100                                                 
   Sold  Validation       Duplicate column names      0       

## Step 11: Export Cleaned Deliverables

Save the cleaned Sold and Listings datasets and the consolidated quality report to the project's `output` folder. These are new files; the input CSVs are not overwritten.

The report consolidates validation results, missing-value counts, and numeric, date, and geographic quality flags.

In [13]:
# Step 11: Export cleaned datasets and the consolidated quality report

import os

sold_output = os.path.join(
    OUTPUT_DIR,
    "CRMLSSold_202401_202604_Residential_Cleaned.csv"
)
listings_output = os.path.join(
    OUTPUT_DIR,
    "CRMLSListing_202401_202604_Residential_Cleaned.csv"
)
report_output = os.path.join(
    OUTPUT_DIR,
    "weeks4_5_data_quality_report.csv"
)

# Write new output files. Original input files remain unchanged.
sold_clean.to_csv(sold_output, index=False)
listings_clean.to_csv(listings_output, index=False)
quality_report.to_csv(report_output, index=False)

print("=== EXPORTED FILES ===")

for path in [sold_output, listings_output, report_output]:
    size_mb = os.path.getsize(path) / (1024 ** 2)
    print(f"{os.path.basename(path)} — {size_mb:,.1f} MB")

print("\n=== EXPORT VALIDATION ===")
for path in [sold_output, listings_output, report_output]:
    print(f"{os.path.basename(path)}: exists = {os.path.exists(path)}")

=== EXPORTED FILES ===
CRMLSSold_202401_202604_Residential_Cleaned.csv — 264.4 MB
CRMLSListing_202401_202604_Residential_Cleaned.csv — 292.7 MB
weeks4_5_data_quality_report.csv — 0.0 MB

=== EXPORT VALIDATION ===
CRMLSSold_202401_202604_Residential_Cleaned.csv: exists = True
CRMLSListing_202401_202604_Residential_Cleaned.csv: exists = True
weeks4_5_data_quality_report.csv: exists = True


## Step 12: Create a Reproducible Cleaning Script

Create `weeks4_5_data_cleaning.py` in the project folder. The script documents the transformation rules, checks the input files, preserves all records, verifies the mortgage-rate field, and writes the cleaned datasets and consolidated report to `output/`.

The script is intended for reproducibility and can later be committed to GitHub without the large MLS CSV files.

In [14]:
script_content = r'''
"""
IDX Exchange — Weeks 4–5 Data Cleaning
--------------------------------------
Reads mortgage-enriched residential MLS datasets, standardizes key data types,
removes verified redundant columns, adds numeric/date/geographic quality flags,
and exports cleaned datasets plus a consolidated quality report.

Raw input CSV files are never overwritten. No rows are automatically removed.
"""

from pathlib import Path
import pandas as pd

BASE_DIR = Path("/content/drive/MyDrive/IDX-Exchange")
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUTS = {
    "Sold": OUTPUT_DIR / "CRMLSSold_202401_202604_Residential_WithMortgageRate.csv",
    "Listings": OUTPUT_DIR / "CRMLSListing_202401_202604_Residential_WithMortgageRate.csv",
}

OUTPUTS = {
    "Sold": OUTPUT_DIR / "CRMLSSold_202401_202604_Residential_Cleaned.csv",
    "Listings": OUTPUT_DIR / "CRMLSListing_202401_202604_Residential_Cleaned.csv",
}
REPORT_PATH = OUTPUT_DIR / "weeks4_5_data_quality_report.csv"

DATE_FIELDS = [
    "CloseDate",
    "PurchaseContractDate",
    "ListingContractDate",
    "ContractStatusChangeDate",
]

NUMERIC_FIELDS = [
    "ClosePrice",
    "ListPrice",
    "LivingArea",
    "DaysOnMarket",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "Latitude",
    "Longitude",
    "rate_30yr_fixed",
]

NUMERIC_RULES = {
    "ClosePrice": ("closeprice_nonpositive_flag", lambda s: s.notna() & (s <= 0)),
    "LivingArea": ("livingarea_nonpositive_flag", lambda s: s.notna() & (s <= 0)),
    "DaysOnMarket": ("daysonmarket_negative_flag", lambda s: s.notna() & (s < 0)),
    "BedroomsTotal": ("bedrooms_negative_flag", lambda s: s.notna() & (s < 0)),
    "BathroomsTotalInteger": ("bathrooms_negative_flag", lambda s: s.notna() & (s < 0)),
}

NUMERIC_FLAGS = [item[0] for item in NUMERIC_RULES.values()]
DATE_FLAGS = [
    "listing_after_close_flag",
    "purchase_after_close_flag",
    "negative_timeline_flag",
]
GEO_FLAGS = [
    "coordinates_missing_flag",
    "latitude_zero_flag",
    "longitude_zero_flag",
    "longitude_positive_flag",
    "coordinates_out_of_range_flag",
    "coordinates_outside_ca_flag",
]


def load_inputs():
    data = {}
    for name, path in INPUTS.items():
        if not path.exists():
            raise FileNotFoundError(f"Required input file not found: {path}")

        print(f"Loading {name}: {path.name}")
        data[name] = pd.read_csv(path, low_memory=False)

        if "rate_30yr_fixed" not in data[name].columns:
            raise ValueError(
                f"{name} input is missing the required rate_30yr_fixed column."
            )

    return data


def remove_verified_listing_duplicates(df):
    """Remove only suffix columns proven equal to their base columns."""
    df = df.copy()

    verified_candidates = [
        "PropertyType.1",
        "ListAgentFirstName.1",
        "DaysOnMarket.1",
        "LivingArea.1",
        "Longitude.1",
        "Latitude.1",
        "ListPrice.1",
        "ListAgentLastName.1",
        "CloseDate.1",
    ]

    for suffix_col in verified_candidates:
        base_col = suffix_col[:-2]

        if suffix_col not in df.columns:
            continue

        if base_col not in df.columns:
            raise ValueError(
                f"Cannot verify {suffix_col}: base column {base_col} is missing."
            )

        if not df[base_col].equals(df[suffix_col]):
            raise ValueError(
                f"{base_col} and {suffix_col} no longer match exactly. "
                "Review the columns before removing either one."
            )

        df = df.drop(columns=[suffix_col])

    # Preserve the canonical address and recover a value from the suffix
    # only if the canonical address is missing.
    if "UnparsedAddress.1" in df.columns:
        if "UnparsedAddress" not in df.columns:
            df = df.rename(columns={"UnparsedAddress.1": "UnparsedAddress"})
        else:
            recover = (
                df["UnparsedAddress"].isna()
                & df["UnparsedAddress.1"].notna()
            )
            df.loc[recover, "UnparsedAddress"] = df.loc[
                recover, "UnparsedAddress.1"
            ]
            df = df.drop(columns=["UnparsedAddress.1"])

    # Keep the unresolved buyer-office discrepancy for review.
    if "BuyerOfficeName.1" in df.columns:
        review_name = "BuyerOfficeName_suffix_review"
        if review_name in df.columns:
            raise ValueError(f"Review column already exists: {review_name}")
        df = df.rename(
            columns={"BuyerOfficeName.1": review_name}
        )

    return df


def standardize_types(df, dataset_name):
    """Convert specified fields and report values that fail conversion."""
    df = df.copy()
    conversion_rows = []

    for col in DATE_FIELDS:
        if col not in df.columns:
            continue

        original = df[col]
        converted = pd.to_datetime(original, errors="coerce")
        failures = int((original.notna() & converted.isna()).sum())
        conversion_rows.append({
            "dataset": dataset_name,
            "column": col,
            "target_type": "datetime",
            "conversion_failures": failures,
            "final_dtype": str(converted.dtype),
        })
        df[col] = converted

    for col in NUMERIC_FIELDS:
        if col not in df.columns:
            continue

        original = df[col]
        converted = pd.to_numeric(original, errors="coerce")
        failures = int((original.notna() & converted.isna()).sum())
        conversion_rows.append({
            "dataset": dataset_name,
            "column": col,
            "target_type": "numeric",
            "conversion_failures": failures,
            "final_dtype": str(converted.dtype),
        })
        df[col] = converted

    return df, conversion_rows


def add_quality_flags(df):
    """Add numeric, date-consistency, and geographic flags."""
    df = df.copy()

    for col, (flag_col, condition) in NUMERIC_RULES.items():
        if col in df.columns:
            df[flag_col] = condition(df[col]).astype(bool)

    listing = df["ListingContractDate"]
    purchase = df["PurchaseContractDate"]
    close = df["CloseDate"]

    df["listing_after_close_flag"] = (
        listing.notna() & close.notna() & (listing > close)
    )
    df["purchase_after_close_flag"] = (
        purchase.notna() & close.notna() & (purchase > close)
    )

    listing_after_purchase = (
        listing.notna() & purchase.notna() & (listing > purchase)
    )
    purchase_after_close = (
        purchase.notna() & close.notna() & (purchase > close)
    )
    df["negative_timeline_flag"] = (
        listing_after_purchase | purchase_after_close
    )

    lat = df["Latitude"]
    lon = df["Longitude"]

    df["coordinates_missing_flag"] = lat.isna() | lon.isna()
    df["latitude_zero_flag"] = lat.notna() & lat.eq(0)
    df["longitude_zero_flag"] = lon.notna() & lon.eq(0)
    df["longitude_positive_flag"] = lon.notna() & lon.gt(0)

    df["coordinates_out_of_range_flag"] = (
        (lat.notna() & ~lat.between(-90, 90))
        | (lon.notna() & ~lon.between(-180, 180))
    )

    # Broad California bounding box; outside does not necessarily mean invalid.
    df["coordinates_outside_ca_flag"] = (
        lat.notna()
        & lon.notna()
        & (
            ~lat.between(32.5, 42.1)
            | ~lon.between(-124.5, -114.0)
        )
    )

    return df


def build_quality_report(originals, cleaned):
    """Build one report covering validation, missingness, and quality flags."""
    rows = []

    def add(dataset, category, check, value, notes=""):
        rows.append({
            "dataset": dataset,
            "category": category,
            "check": check,
            "value": value,
            "notes": notes,
        })

    for name, df in cleaned.items():
        original = originals[name]

        add(name, "Validation", "Original row count", len(original))
        add(
            name, "Validation", "Cleaned row count", len(df),
            "Rows retained; no row-level filtering performed."
        )
        add(
            name, "Validation", "Row count unchanged",
            len(original) == len(df)
        )
        add(name, "Validation", "Cleaned column count", len(df.columns))
        add(
            name, "Validation", "Duplicate column names",
            int(df.columns.duplicated().sum())
        )
        add(name, "Validation", "Exact duplicate rows", int(df.duplicated().sum()))
        add(
            name, "Validation", "Mortgage rate field present",
            "rate_30yr_fixed" in df.columns
        )
        add(
            name, "Validation", "Missing mortgage rate values",
            int(df["rate_30yr_fixed"].isna().sum())
        )

        fields = [
            "CloseDate", "PurchaseContractDate", "ListingContractDate",
            "ContractStatusChangeDate", "ClosePrice", "ListPrice",
            "LivingArea", "DaysOnMarket", "BedroomsTotal",
            "BathroomsTotalInteger", "Latitude", "Longitude",
            "rate_30yr_fixed",
        ]
        for col in fields:
            if col in df.columns:
                missing = int(df[col].isna().sum())
                add(
                    name, "Missingness", col, missing,
                    f"{missing / len(df) * 100:.2f}% of rows missing."
                )

        for flag in NUMERIC_FLAGS:
            if flag in df.columns:
                add(
                    name, "Numeric quality", flag, int(df[flag].sum()),
                    "Flagged rows; records retained."
                )

        for flag in DATE_FLAGS:
            add(
                name, "Date consistency", flag, int(df[flag].sum()),
                "Flagged rows; counts may overlap."
            )

        for flag in GEO_FLAGS:
            add(
                name, "Geographic quality", flag, int(df[flag].sum()),
                "Flagged rows; counts may overlap."
            )

        for col in DATE_FIELDS:
            if col in df.columns:
                add(
                    name, "Data types", f"{col} dtype", str(df[col].dtype),
                    "Expected datetime dtype."
                )

        for category, flags in [
            ("Numeric quality", NUMERIC_FLAGS),
            ("Date consistency", DATE_FLAGS),
            ("Geographic quality", GEO_FLAGS),
        ]:
            available = [flag for flag in flags if flag in df.columns]
            add(
                name, "Record-level summary",
                f"Rows with any {category.lower()} flag",
                int(df[available].any(axis=1).sum())
            )

    return pd.DataFrame(rows)


def main():
    originals = load_inputs()
    cleaned = {}
    conversion_rows = []

    for name, original in originals.items():
        print(f"\nProcessing {name}: {len(original):,} input rows")

        df = original.copy()

        if name == "Listings":
            df = remove_verified_listing_duplicates(df)

        df, conversions = standardize_types(df, name)
        conversion_rows.extend(conversions)
        df = add_quality_flags(df)

        if len(df) != len(original):
            raise AssertionError(f"{name}: row count changed unexpectedly.")
        if df.columns.duplicated().any():
            raise AssertionError(f"{name}: duplicate column names remain.")
        if "rate_30yr_fixed" not in df.columns:
            raise AssertionError(f"{name}: mortgage-rate field missing.")

        cleaned[name] = df

    conversion_report = pd.DataFrame(conversion_rows)
    print("\n=== TYPE CONVERSION CHECK ===")
    print(conversion_report.to_string(index=False))

    total_failures = int(conversion_report["conversion_failures"].sum())
    print(f"\nTotal conversion failures: {total_failures:,}")

    report = build_quality_report(originals, cleaned)

    for name, df in cleaned.items():
        df.to_csv(OUTPUTS[name], index=False)
        print(
            f"Exported {OUTPUTS[name].name}: "
            f"{len(df):,} rows, {len(df.columns):,} columns"
        )

    report.to_csv(REPORT_PATH, index=False)
    print(f"Exported {REPORT_PATH.name}: {len(report):,} report rows")

    print("\n=== FINAL SUMMARY ===")
    for name, df in cleaned.items():
        print(
            f"{name}: rows={len(df):,}, columns={len(df.columns):,}, "
            f"exact_duplicate_rows={df.duplicated().sum():,}, "
            f"missing_mortgage_rates={df['rate_30yr_fixed'].isna().sum():,}"
        )

    if total_failures:
        print(
            "\nWARNING: Some values failed type conversion. "
            "Review the conversion report before treating the outputs as final."
        )


if __name__ == "__main__":
    main()
'''

script_path = os.path.join(BASE_DIR, "weeks4_5_data_cleaning.py")

with open(script_path, "w", encoding="utf-8") as f:
    f.write(script_content.strip() + "\n")

print("Script created:", script_path)
print("File exists:", os.path.exists(script_path))
print("File size:", f"{os.path.getsize(script_path):,} bytes")

Script created: /content/drive/MyDrive/IDX-Exchange/weeks4_5_data_cleaning.py
File exists: True
File size: 12,732 bytes


In [15]:
with open(script_path, "r", encoding="utf-8") as f:
    saved_script = f.read()

checks = {
    "Input file definitions": "WithMortgageRate.csv" in saved_script,
    "Redundant-column verification": "remove_verified_listing_duplicates" in saved_script,
    "Date and numeric conversions": "standardize_types" in saved_script,
    "Numeric/date/geographic flags": "add_quality_flags" in saved_script,
    "Consolidated quality report": "build_quality_report" in saved_script,
    "Export logic": "to_csv" in saved_script,
}

print("=== SCRIPT CONTENT CHECK ===")
for check, passed in checks.items():
    print(f"{check}: {'PASS' if passed else 'FAIL'}")

=== SCRIPT CONTENT CHECK ===
Input file definitions: PASS
Redundant-column verification: PASS
Date and numeric conversions: PASS
Numeric/date/geographic flags: PASS
Consolidated quality report: PASS
Export logic: PASS


## Step 13: Reproducibility Test

Run the saved Python script as a standalone program. Confirm that it loads the enriched inputs, performs the documented transformations, and regenerates the cleaned datasets and quality report without errors.

In [16]:
import subprocess
import os

result = subprocess.run(
    ["python", script_path],
    capture_output=True,
    text=True,
    timeout=1800
)

print("=== SCRIPT EXECUTION ===")
print("Return code:", result.returncode)

print("\n=== SCRIPT OUTPUT ===")
print(result.stdout[-12000:] if result.stdout else "(No standard output)")

if result.stderr:
    print("\n=== SCRIPT ERRORS / WARNINGS ===")
    print(result.stderr[-6000:])

if result.returncode == 0:
    print("\nSUCCESS: The standalone script completed.")
else:
    print("\nFAILED: Review the output above before proceeding.")

=== SCRIPT EXECUTION ===
Return code: 0

=== SCRIPT OUTPUT ===
Loading Sold: CRMLSSold_202401_202604_Residential_WithMortgageRate.csv
Loading Listings: CRMLSListing_202401_202604_Residential_WithMortgageRate.csv

Processing Sold: 414,003 input rows

Processing Listings: 547,162 input rows

=== TYPE CONVERSION CHECK ===
 dataset                   column target_type  conversion_failures    final_dtype
    Sold                CloseDate    datetime                    0 datetime64[ns]
    Sold     PurchaseContractDate    datetime                    0 datetime64[ns]
    Sold      ListingContractDate    datetime                    0 datetime64[ns]
    Sold ContractStatusChangeDate    datetime                    0 datetime64[ns]
    Sold               ClosePrice     numeric                    0        float64
    Sold                ListPrice     numeric                    0        float64
    Sold               LivingArea     numeric                    0        float64
    Sold             Da

### Final Output Verification
Confirm that both cleaned datasets and the consolidated quality report were saved successfully, with mortgage rates and quality flags retained.

In [17]:
import os
import pandas as pd

files_to_check = {
    "Sold cleaned dataset": os.path.join(
        OUTPUT_DIR, "CRMLSSold_202401_202604_Residential_Cleaned.csv"
    ),
    "Listings cleaned dataset": os.path.join(
        OUTPUT_DIR, "CRMLSListing_202401_202604_Residential_Cleaned.csv"
    ),
    "Quality report": os.path.join(
        OUTPUT_DIR, "weeks4_5_data_quality_report.csv"
    ),
}

for name, path in files_to_check.items():
    print(f"\n{name}")
    print("Exists:", os.path.exists(path))

    if os.path.exists(path):
        print(f"Size: {os.path.getsize(path) / (1024**2):.1f} MB")

        preview = pd.read_csv(path, nrows=3)
        print("Columns:", len(preview.columns))
        print("Has mortgage rate:", "rate_30yr_fixed" in preview.columns)
        print("Quality flags:", [
            c for c in preview.columns if c.endswith("_flag")
        ][:5])


Sold cleaned dataset
Exists: True
Size: 264.4 MB
Columns: 100
Has mortgage rate: True
Quality flags: ['closeprice_nonpositive_flag', 'livingarea_nonpositive_flag', 'daysonmarket_negative_flag', 'bedrooms_negative_flag', 'bathrooms_negative_flag']

Listings cleaned dataset
Exists: True
Size: 292.7 MB
Columns: 90
Has mortgage rate: True
Quality flags: ['closeprice_nonpositive_flag', 'livingarea_nonpositive_flag', 'daysonmarket_negative_flag', 'bedrooms_negative_flag', 'bathrooms_negative_flag']

Quality report
Exists: True
Size: 0.0 MB
Columns: 5
Has mortgage rate: False
Quality flags: []
